In [1]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
from langchain_classic.chains import create_retrieval_chain
from dotenv import load_dotenv

print("Imports successful! Ready to build the RAG chain.")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful! Ready to build the RAG chain.


In [2]:
load_dotenv()

True

In [14]:
docs=[
    "Company Policy 402A: Remote Work. Employees in the Engineering department are allowed to work remotely up to 3 days a week. However, during the final two weeks of a product launch cycle (known as the 'Code Red' phase), all Engineering staff must be on-site. Marketing department employees are fully remote.",
    "Expense Policy: Meals and Entertainment. The company will reimburse up to $75 per day for meals while traveling for business. Alcohol is strictly non-reimbursable under any circumstances, unless explicitly pre-approved by a VP-level executive in writing. Use billing code EXP-99 for standard meal submissions.",
    "Employee Directory Snapshot - Engineering: John Doe (ID: EMP-101, Role: Lead Architect). Jane Smith (ID: EMP-102, Role: Backend Developer). Michael Chang (ID: EMP-103, Role: AI Engineer).",
    "Hardware and IT Support: If a company laptop is malfunctioning and displaying error code ERR-909 (Motherboard failure) or ERR-404 (Network card), do not attempt to fix it yourself. Submit a critical ticket to the IT portal using form IT-REQ-V2. For minor issues like battery replacement, contact the facilities manager, David Lee.",
    "Leave Policy: PTO and Sick Leave. Full-time employees accrue 15 days of Paid Time Off (PTO) annually. Up to 5 unused PTO days can be rolled over into the next calendar year, but these rollover days expire automatically on March 31st. Sick leave is unlimited but requires a doctor's note after 3 consecutive days."
]

In [15]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

In [16]:
system_prompt = (
    "You are an AI assistant for a corporate HR department. "
    "Use the following pieces of retrieved context to answer the user's question. "
    "If the answer is not in the context, say 'I do not have that information.' Do not guess.\n\n"
    "Context:\n{context}"
)

In [17]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [18]:
document_chain = create_stuff_documents_chain(llm, prompt)

In [19]:
embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7119.91it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
vectorstore = Chroma.from_texts(texts=docs, embedding = embed_model)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [21]:
keyword_retriever = BM25Retriever.from_texts(docs)
keyword_retriever.k = 2

In [22]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, keyword_retriever],
    weights=[0.5, 0.5]
)

In [23]:
rag_chain = create_retrieval_chain(hybrid_retriever, document_chain)

In [24]:
response = rag_chain.invoke({"input": "What is the employee ID for Jane Doe?"})

In [25]:
print(response)

{'input': 'What is the employee ID for Jane Doe?', 'context': [Document(metadata={}, page_content=' '), Document(metadata={}, page_content='Expense Policy: Meals and Entertainment. The company will reimburse up to $75 per day for meals while traveling for business. Alcohol is strictly non-reimbursable under any circumstances, unless explicitly pre-approved by a VP-level executive in writing. Use billing code EXP-99 for standard meal submissions.'), Document(metadata={}, page_content='Employee Directory Snapshot - Engineering: John Doe (ID: EMP-101, Role: Lead Architect). Jane Smith (ID: EMP-102, Role: Backend Developer). Michael Chang (ID: EMP-103, Role: AI Engineer).')], 'answer': 'I do not have that information.'}


In [26]:
response = rag_chain.invoke({"input": "What is the employee ID for Jane Smith?"})
print(response["answer"])

Jane Smith's employee ID is EMP-102.


In [27]:
response = rag_chain.invoke({"input": "I am an AI Engineer. We are one week away from our product launch. Can I work from home tomorrow?"})
print(response["answer"])

Based on the provided context, Company Policy 402A states that during the final two weeks of a product launch cycle (known as the 'Code Red' phase), all Engineering staff must be on-site. Since you are one week away from the product launch, you are currently in the 'Code Red' phase. Therefore, you are required to be on-site tomorrow.
